# Pocket checks of the conformational states: raw vs minimized models

Written 2026-11-23, branch `tm-states`. This notebook records the first full
run of the pocket check on raw and OpenMM-minimized models, so the results and
their reading are not lost. Every number below is recomputed from the pipeline
outputs in `results/`; the prose quotes the values of that first run.

**What is being checked.** `tm_states` (step 5) labels each model OF / Occ / IF
from the extremities of the TM-Cα PCA cloud, with a barycentric weight α for
each state, and exports 20 models per state spread over the α range of that
state's members. `tm_cavity` (step 6) runs pyKVFinder on those 60 models per
protein, takes the cavity lined by the most CDD substrate-pocket residues
(InterPro), and calls it *extra-open*, *intra-open*, *occluded*, *both-open*,
*lateral-only* or *no pocket*. A model **agrees** when OF → extra-open,
Occ → occluded, IF → intra-open.

`tm_minimize` (step 5b, port of NanoPulldown's scheme) minimizes the same
exported models (PDBFixer, vacuum pre-relax, amber14 + GBn2), and
`tm_cavity_minimized` repeats the pocket check on them. The question: the pocket
calls depend on side-chain positions, so how much does minimization change them?

Setup of this run (from `config.yaml`): pyKVFinder probe in 1.4 Å, **probe out
8 Å** (ChimeraX's default of 4 Å turns wide-open vestibules into bulk solvent
and loses the OF pockets), volume cutoff 100 Å³, min opening area 20 Å²;
minimization: max 2000 steps, 500 pre-relax steps.

Environment: `nanopulldown-cavity-notebook` (pandas, pyarrow, matplotlib).

In [1]:
import glob, json, os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml

# run from the repo root whether the kernel starts in notebooks/ or not
if Path.cwd().name == "notebooks":
    os.chdir("..")
pd.set_option("display.width", 160)

cfg = yaml.safe_load(open("config.yaml"))
{k: cfg.get(k) for k in ("tm_states", "tm_cavity", "tm_minimize")}

{'tm_states': {'mass_frac': 0.005,
  'radius_frac': 0.05,
  'n_export': 20,
  'max_betweenness': 0.5,
  'min_flatness': 0.1,
  'min_gap': 5.0},
 'tm_cavity': {'volume_cutoff': 100,
  'probe_out': 8.0,
  'min_area': 20,
  'side_z': 0.5,
  'min_cdd_frac': 0.4},
 'tm_minimize': {'platforms': ['CUDA', 'OpenCL'],
  'max_steps': 2000,
  'probe_steps': 200,
  'pre_relax_steps': 500,
  'max_minutes': 15}}

In [2]:
def load(name):
    rows = []
    for f in sorted(glob.glob(f"results/*/tm_states/{name}.parquet")):
        d = pd.read_parquet(f)
        js = json.load(open(Path(f).parent / "tm_states.json"))
        d["verdict"] = js["verdict"]
        rows.append(d)
    d = pd.concat(rows, ignore_index=True)
    d["resolved"] = d.verdict.eq("resolved")
    return d

mini = load("minimize")
raw = load("cavity")
mnz = load("cavity_minimized")
key = ["protein", "state", "model_id", "depth"]
m = raw.merge(mnz.drop(columns=["verdict", "resolved"]), on=key, suffixes=("_raw", "_min"))
print(f"{mini.protein.nunique()} proteins, {len(mini)} minimized models; "
      f"{len(raw)} raw / {len(mnz)} minimized pocket checks, {len(m)} paired")
print(raw.drop_duplicates("protein").verdict.value_counts().to_string())

53 proteins, 3180 minimized models; 3180 raw / 3180 minimized pocket checks, 3180 paired
verdict
resolved      39
unresolved    14


## 1. Minimization

**Finding (first run):** all 3,180 models (53 proteins × 60) minimized and
converged, about 4 s each on the M4 Max GPU (OpenMM OpenCL platform, device
"Apple M4 Max", single precision). Models moved very little: median RMSD
0.30 Å over Cα and 0.41 Å over heavy atoms, maximum 1.4 Å and 1.9 Å. The
backbone barely moves, so any change in the pocket calls comes from side chains.

In [3]:
print(mini.status.value_counts().to_string(), "\nconverged:", f"{mini.converged.mean():.1%}",
      "\nplatform:", mini.platform.value_counts().to_dict())
mini[["seconds", "steps", "rmsd_ca", "rmsd_heavy"]].describe().round(2).T

status
ok    3180 
converged: 100.0% 
platform: {'OpenCL': 3180}


,count,mean,std,min,25%,50%,75%,max
seconds,3180.0,4.15,0.61,2.66,3.76,4.02,4.44,9.66
steps,3180.0,250.69,22.70,200.00,250.00,250.00,250.00,400.00
rmsd_ca,3180.0,0.31,0.05,0.21,0.27,0.30,0.34,1.37
rmsd_heavy,3180.0,0.48,0.20,0.29,0.37,0.41,0.47,1.85


In [4]:
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
for a, c, lab in zip(ax, ["rmsd_ca", "rmsd_heavy"], ["Cα", "heavy atoms"]):
    a.hist(mini[c], bins=60, color="0.4")
    a.set_xlabel(f"RMSD to raw model, {lab} (Å)"); a.set_yscale("log")
ax[0].set_ylabel("models")
fig.tight_layout()

## 2. Agreement with the PCA state, raw → minimized

**Finding (first run):**

| | OF | Occ | IF | All |
|---|---|---|---|---|
| Resolved (39 proteins) | 98.6 → 95.4 % | 33.5 → 39.7 % | 85.3 → 82.4 % | 72.4 → 72.5 % |
| Unresolved (14 proteins) | 53.6 → 52.9 % | 11.8 → 14.6 % | 63.9 → 61.8 % | 43.1 → 43.1 % |

Minimization changes little overall. The unresolved proteins agree about as
well as chance, consistent with their verdict: their corners are not real states.

In [5]:
def agreement(d):
    t = d.groupby(["resolved", "state"])[["agree_raw", "agree_min"]].mean().unstack("state")
    for c in ("agree_raw", "agree_min"):
        t[(c, "All")] = d.groupby("resolved")[c].mean()
    t = (100 * t).round(1)
    return t.reindex(columns=[(c, s) for c in ("agree_raw", "agree_min")
                              for s in ("OF", "Occ", "IF", "All")])
agreement(m)

agree_raw                   agree_min                  
state           OF   Occ    IF   All        OF   Occ    IF   All
resolved                                                        
False         53.6  11.8  63.9  43.1      52.9  14.6  61.8  43.1
True          98.6  33.5  85.3  72.4      95.4  39.7  82.4  72.5

## 3. What minimization changes

**Finding (first run):** only 6 % of the calls change in the resolved proteins.
The main shift is 66 models going **intra-open → occluded**: side chains close a
narrow inside opening. That helps Occ (+6 points) but costs IF (−3). OF loses
about 3 points: a few extra-open models turn lateral-only, no pocket, or
occluded.

In [6]:
r = m[m.resolved]
print(f"calls changed (resolved): {(r.call_raw != r.call_min).mean():.1%}")
pd.crosstab(r.call_raw, r.call_min, rownames=["raw"], colnames=["minimized"])

calls changed (resolved): 6.0%


minimized,both-open,extra-open,intra-open,lateral-only,no pocket,occluded
raw,,,,,,
both-open,10,3,3,0,0,1
extra-open,1,1061,4,14,12,17
intra-open,1,2,799,0,0,66
no pocket,0,0,0,0,45,0
occluded,0,0,3,0,13,285


In [7]:
pd.crosstab(r.state, r.call_min, rownames=["state"], colnames=["minimized call"])

minimized call,both-open,extra-open,intra-open,lateral-only,no pocket,occluded
state,,,,,,
IF,8,27,643,0,49,53
OF,4,744,5,10,11,6
Occ,0,295,161,4,10,310


## 4. Occ disagreement comes from which models were exported

**Finding (first run):** within the Occ models of resolved proteins, agreement
follows the occluded weight α. Models near the occluded tip (α > 0.9) are
occluded 93–94 % of the time; most exported Occ models sit at α 0.5–0.7 and
agree only 14 % (raw) / 23 % (minimized). They lie partway along an arm and leak
to the outside or the inside.

So the OF and IF corners check out, and the true occluded tip is occluded. The
weak Occ score is mainly a property of the export (spread over the α range of
the members), not of the corner itself. → next: α thresholding.

In [8]:
bins = [0, .5, .7, .9, 1.0001]
def by_alpha(d):
    d = d.assign(alpha_bin=pd.cut(d.alpha, bins))
    g = d.groupby(["state", "alpha_bin"], observed=True)
    return pd.concat({"n": g.size(),
                      "agree raw %": (100 * g.agree_raw.mean()).round(0),
                      "agree min %": (100 * g.agree_min.mean()).round(0)}, axis=1)
r = r.rename(columns={"alpha_raw": "alpha"})
by_alpha(r)

n  agree raw %  agree min %
state alpha_bin                                
IF    (0.0, 0.5]   28         61.0         54.0
      (0.5, 0.7]  329         86.0         81.0
      (0.7, 0.9]  335         90.0         89.0
      (0.9, 1.0]   88         74.0         73.0
OF    (0.0, 0.5]   41         93.0         88.0
      (0.5, 0.7]  353         99.0         97.0
      (0.7, 0.9]  298         99.0         94.0
      (0.9, 1.0]   88         98.0         97.0
Occ   (0.0, 0.5]   51         10.0         14.0
      (0.5, 0.7]  454         14.0         23.0
      (0.7, 0.9]  166         55.0         59.0
      (0.9, 1.0]  109         94.0         93.0

In [9]:
fig, ax = plt.subplots(1, 3, figsize=(12, 3.2), sharey=True)
for a, s in zip(ax, ["OF", "Occ", "IF"]):
    d = r[r.state == s]
    e = np.linspace(d.alpha.min(), 1, 9)
    c = pd.cut(d.alpha, e, include_lowest=True)
    for col, lab, st in [("agree_raw", "raw", "o-"), ("agree_min", "minimized", "s--")]:
        y = d.groupby(c, observed=True)[col].mean()
        a.plot([iv.mid for iv in y.index], y.values, st, label=lab)
    a2 = a.twinx(); a2.hist(d.alpha, bins=e, color="0.85", zorder=0); a2.set_yticks([])
    a.set_zorder(a2.get_zorder() + 1); a.patch.set_visible(False)
    a.set_title(f"{s} (n={len(d)})"); a.set_xlabel("α of the exported model")
ax[0].set_ylabel("agreement with pocket call"); ax[0].legend()
fig.suptitle("Resolved proteins; grey = how many exported models at that α")
fig.tight_layout()


### 4b. How many models would an α floor keep?

For the α-thresholding discussion: count, per protein and state, the **members**
of the state (all fitted models assigned to it, not only the 20 exported) whose
α for that state is at least 0.7, 0.8 or 0.9. A floor leaves a folder thin when
fewer than `n_export` (20) members pass.

The last columns give the agreement of the already-exported models that pass
the same floor: a preview of what the floor buys, with the caveat that it is
measured on the current export (spread over the α range), not on a new one.


In [10]:

thresholds = [0.7, 0.8, 0.9]
n_export = cfg["tm_states"]["n_export"]

ts = []
for f in sorted(glob.glob("results/*/tm_states/tm_states.parquet")):
    d = pd.read_parquet(f)
    d["protein"] = Path(f).parent.parent.name
    d["verdict"] = json.load(open(Path(f).parent / "tm_states.json"))["verdict"]
    ts.append(d)
ts = pd.concat(ts, ignore_index=True)
ts = ts[ts.state.isin(["OF", "Occ", "IF"])]
ts["alpha"] = [a[f"alpha_{s}"] for s, (_, a) in zip(ts.state, ts.iterrows())]

counts = (ts.groupby(["verdict", "protein", "state"]).alpha
            .agg(members="size", **{f">={t}": (lambda x, t=t: int((x >= t).sum())) for t in thresholds})
            .reset_index())

rows = []
for (v, s), g in counts.groupby(["verdict", "state"]):
    e = m[(m.verdict == v) & (m.state == s)]
    for t in thresholds:
        n = g[f">={t}"]
        ee = e[e.alpha_raw >= t]
        rows.append({"verdict": v, "state": s, "floor": t,
                     "median members": int(n.median()),
                     f"proteins >= {n_export}": int((n >= n_export).sum()),
                     "proteins 1-19": int(((n > 0) & (n < n_export)).sum()),
                     "proteins 0": int((n == 0).sum()),
                     "exported passing": len(ee),
                     "agree raw %": round(100 * ee.agree_raw.mean()) if len(ee) else np.nan,
                     "agree min %": round(100 * ee.agree_min.mean()) if len(ee) else np.nan})
pd.DataFrame(rows).set_index(["verdict", "state", "floor"])


median members  proteins >= 20  proteins 1-19  proteins 0  exported passing  agree raw %  agree min %
verdict    state floor                                                                                                       
resolved   IF    0.7               704              39              0           0               423           87           86
                 0.8               304              39              0           0               239           83           81
                 0.9                83              39              0           0                88           74           73
           OF    0.7               443              39              0           0               386           99           95
                 0.8               198              39              0           0               210           98           95
                 0.9                64              39              0           0                88           98           97
           Occ   0.7               303              39              0           0               275           70           72
                 0.8               188              39              0           0               183           87           87
                 0.9                99              39              0           0               109           94           93
unresolved IF    0.7               736              14              0           0               190           62           62
                 0.8               478              14              0           0               151           61           63
                 0.9                86              14              0           0                83           59           60
           OF    0.7               143              14              0           0               184           43           42
                 0.8               122              14              0           0               145           35           34
                 0.9                71              14              0           0                90           30           32
           Occ   0.7               314              14              0           0               125            6           11
                 0.8               238              14              0           0                72            7           15
                 0.9                78              14              0           0                32           12           12

Per protein (resolved first), sorted by the Occ members at α ≥ 0.8.

In [11]:

pc = counts.pivot_table(index=["verdict", "protein"], columns="state",
                        values=["members"] + [f">={t}" for t in thresholds])
pc = pc.reindex(columns=[(k, s) for s in ("OF", "Occ", "IF") for k in ["members"] + [f">={t}" for t in thresholds]])
pc.columns = [f"{s} {k}" for k, s in pc.columns]
pc = pc.fillna(0).astype(int)
pc.sort_values(["verdict", "Occ >=0.8"]).style.map(
    lambda x: "background-color: #f6d5d5" if x < n_export else "", subset=[c for c in pc.columns if ">=" in c])


## 5. Per protein

Agreement per protein and state (resolved proteins), raw and minimized, sorted
by the minimized Occ agreement.

In [12]:
pp = (100 * r.groupby(["protein", "state"])[["agree_raw", "agree_min"]].mean()).round(0).unstack("state")
pp.columns = [f"{s} {c.split('_')[1]}" for c, s in pp.columns]
pp = pp[[f"{s} {k}" for s in ("OF", "Occ", "IF") for k in ("raw", "min")]]
pp.sort_values("Occ min")

,OF raw,OF min,Occ raw,Occ min,IF raw,IF min
protein,,,,,,
NPF4.1_Q9LSF0,90.0,90.0,0.0,0.0,65.0,65.0
NPF8.2_Q9LFB8,100.0,100.0,5.0,5.0,100.0,100.0
NPF4.4_Q56XQ6,100.0,90.0,15.0,5.0,95.0,95.0
NPF4.3_Q93VV5,100.0,100.0,0.0,5.0,100.0,100.0
NPF8.3_P46032,100.0,100.0,10.0,10.0,100.0,100.0
NPF5.11_Q8RX67,100.0,100.0,10.0,10.0,100.0,100.0
NPF8.1_Q9M390,100.0,100.0,10.0,10.0,100.0,100.0
NPF4.5_Q8VYE4,95.0,95.0,10.0,10.0,95.0,95.0
NPF5.8_Q9LFR1,95.0,95.0,5.0,15.0,95.0,95.0


## Open questions

- **α thresholding of the export** (to discuss): an α floor for Occ (≈ 0.8?),
  or take the top-α members instead of spreading over the range. Check what it
  does to the number of proteins that still have 20 Occ models.
- Should the pocket call feed back into the annotation (e.g. flag Occ corners
  whose top-α models are not occluded)?